# BERT 中文金融情感分类 v2
用 DeepSeek 三只股票打标数据（比亚迪+亿纬锂能+国轩高科）微调 BERT，三分类（负面/中性/正面）

## 1. 安装依赖

In [ ]:
!pip install transformers datasets accelerate scikit-learn

## 2. 上传数据
运行后选择 merge.csv（三只股票合并后的DeepSeek打分文件）

In [ ]:
from google.colab import files
uploaded = files.upload()
print(f"已上传 {len(uploaded)} 个文件")

## 3. 读取数据

In [ ]:
import pandas as pd
from collections import Counter
import glob

filename = glob.glob('*.csv')[0]
# 先试 gbk，不行用 gb18030（更全的中文编码）
try:
    df = pd.read_csv(filename, encoding='gbk')
except UnicodeDecodeError:
    df = pd.read_csv(filename, encoding='gb18030')

df = df[df['text'].notna() & (df['text'].str.len() >= 3)]
df['label'] = df['3sentiment'].astype(int)
df['label'] = df['label'].map({-1: 0, 0: 1, 1: 2})

print(f'总条数: {len(df)}')
print(f'标签分布:')
names = {0:'负面', 1:'中性', 2:'正面'}
for k, v in sorted(Counter(df['label']).items()):
    print(f'  {names[k]}: {v}')
df[['text','label']].head()

## 4. 划分训练集/验证集

In [ ]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'].tolist(), df['label'].tolist(),
    test_size=0.15, random_state=42, stratify=df['label']
)
print(f'训练集: {len(train_texts)} 条')
print(f'验证集: {len(val_texts)} 条')

## 5. 加载 BERT 并分词

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = 'bert-base-chinese'

tokenizer = AutoTokenizer.from_pretrained(model_name)
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)

In [ ]:
import torch

class AlbertDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = AlbertDataset(train_encodings, train_labels)
val_dataset = AlbertDataset(val_encodings, val_labels)

## 6. 微调 BERT

In [ ]:
from transformers import Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1': f1_score(labels, predictions, average='weighted')
    }

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=15,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=3e-5,
    warmup_steps=200,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

model.save_pretrained('./bert_finance_model')
tokenizer.save_pretrained('./bert_finance_model')
print('训练完成，模型已保存')

## 7. 验证集效果

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

preds = trainer.predict(val_dataset)
y_pred = np.argmax(preds.predictions, axis=1)
y_true = preds.label_ids

print(classification_report(y_true, y_pred, target_names=['负面','中性','正面']))
print('混淆矩阵:')
print(confusion_matrix(y_true, y_pred))

## 8. 保存模型到本地

In [ ]:
import shutil
shutil.make_archive('model', 'zip', './bert_finance_model')
files.download('model.zip')
print('model.zip 已下载')

## 9. 快速测试

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AutoModelForSequenceClassification.from_pretrained('./bert_finance_model').to(device)
tokenizer = AutoTokenizer.from_pretrained('./bert_finance_model')

model.eval()
tests = [
    "今天涨停了，太爽了！",
    "完了完了，跌停了，血亏",
    "今天成交量还可以，观望一下",
    "宁德时代就是垃圾，天天跌",
    "比亚迪牛逼，继续冲高！",
]
for t in tests:
    inputs = tokenizer(t, return_tensors='pt', truncation=True, padding=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    prob = torch.softmax(logits, dim=1).cpu().numpy()[0]
    pred = ['负面','中性','正面'][prob.argmax()]
    print(f'{pred}\t{prob}\t{t}')